### Example conversion notebook to Automodel's image retrieval dataset format

This notebook is provided as an example of how to create a training dataset to start training of embedding models for visual document retrieval. Users are responsible for validating the quality of the generated dataset samples.

In [ ]:
import os
from pathlib import Path

#### Setup

In [ ]:
# SET HERE a filesystem path for dowloading the datasets from HF Hub, and for saving the converted train set and corpus
OUTPUT_ROOT = Path("<DATASETS_PATH>/automodel_colpali")

In [ ]:
# HF datasets with the corpus and queries - Do not change
QUERY_REPO_ID = "Tevatron/colpali"
CORPUS_REPO_ID = "Tevatron/colpali-corpus"
SPLIT = "train"

In [ ]:
TRAIN_JSON_PATH = OUTPUT_ROOT / "training_datasets" / "colpali_train.json"
CORPUS_DIR = OUTPUT_ROOT / "corpus" / "colpali_train_set"
HF_CACHE_DIR = OUTPUT_ROOT / "hf_cache"
CORPUS_ID = "colpali_train_set"
CORPUS_NUM_SHARDS = 80

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE_DIR / "datasets")

TRAIN_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training JSON output: {TRAIN_JSON_PATH}")
print(f"Corpus output: {CORPUS_DIR}")
print(f"HuggingFace cache: {HF_CACHE_DIR}")

In [ ]:
import json
from typing import Any

from datasets import load_dataset

try:
    from tqdm.auto import tqdm
except ImportError:

    def tqdm(iterable, **kwargs):
        """Dummy tqdm implementation"""
        return iterable


QUERY_REQUIRED_COLUMNS = {
    "query_id",
    "query_text",
    "positive_document_ids",
    "negative_document_ids",
}
CORPUS_REQUIRED_COLUMNS = {"docid", "image", "text", "source"}


def normalize_doc_ids(value: Any) -> list[str]:
    """Normalize doc_ids to a list of strings"""
    if value is None:
        return []
    if isinstance(value, list):
        return [str(doc_id) for doc_id in value]
    return [str(value)]


def make_image_filename_from_docid(docid: Any) -> str:
    """Make image_filename from docid"""
    if docid is None:
        raise ValueError("Corpus row has docid=None; cannot derive image_filename")
    image_filename = str(docid)
    if not image_filename:
        raise ValueError("Corpus row has empty docid; cannot derive image_filename")
    return image_filename


def map_document_ids(
    doc_ids: list[str], docid_to_image_filename: dict[str, str]
) -> tuple[list[dict[str, str]], list[str]]:
    """Map document_ids to image_filename"""
    mapped_docs = []
    missing_doc_ids = []
    for doc_id in doc_ids:
        image_filename = docid_to_image_filename.get(doc_id)
        if image_filename is None:
            missing_doc_ids.append(doc_id)
        else:
            mapped_docs.append({"id": image_filename})
    return mapped_docs, missing_doc_ids

#### Loading the queries with positive and negative pages mapping from corpus

In [ ]:
query_ds = load_dataset(QUERY_REPO_ID, split=SPLIT, cache_dir=str(HF_CACHE_DIR / "datasets"))
print(query_ds)
print(f"Query columns: {query_ds.column_names}")

In [ ]:
corpus_ds = load_dataset(CORPUS_REPO_ID, split=SPLIT, cache_dir=str(HF_CACHE_DIR / "datasets"))
print(corpus_ds)
print(f"Corpus columns: {corpus_ds.column_names}")

#### Checking one example query with positive and negative pages

In [ ]:
query_ds[4]

In [ ]:
pos_id = query_ds[4]["positive_document_ids"][0]
corpus_ds[int(pos_id)]["image"]

In [ ]:
pos_id = query_ds[4]["negative_document_ids"][0]
corpus_ds[int(pos_id)]["image"]

#### Converting the corpus

In [ ]:
missing_query_columns = QUERY_REQUIRED_COLUMNS - set(query_ds.column_names)
missing_corpus_columns = CORPUS_REQUIRED_COLUMNS - set(corpus_ds.column_names)
if missing_query_columns:
    raise ValueError(f"Missing query columns: {sorted(missing_query_columns)}")
if missing_corpus_columns:
    raise ValueError(f"Missing corpus columns: {sorted(missing_corpus_columns)}")

In [ ]:
def add_image_filename(batch: dict[str, list[Any]]) -> dict[str, list[str]]:
    """Add image_filename from docid"""
    return {"image_filename": [make_image_filename_from_docid(docid) for docid in batch["docid"]]}


if "image_filename" in corpus_ds.column_names:
    corpus_ds = corpus_ds.remove_columns("image_filename")
corpus_ds = corpus_ds.map(add_image_filename, batched=True, desc="Adding image_filename from docid")

docids = [str(docid) for docid in corpus_ds["docid"]]
image_filenames = [str(image_filename) for image_filename in corpus_ds["image_filename"]]

if len(set(docids)) != len(docids):
    raise ValueError("Corpus docid values must be unique")
if len(set(image_filenames)) != len(image_filenames):
    raise ValueError("Corpus image_filename values must be unique")
if image_filenames != docids:
    raise ValueError("Converted corpus image_filename values must exactly match docid values")

docid_to_image_filename = dict(zip(docids, image_filenames))

print(f"Indexed {len(docid_to_image_filename):,} corpus documents")
print("Sample corpus mapping:")
for docid in docids[:3]:
    print(f"  {docid!r} -> {docid_to_image_filename[docid]!r}")

In [ ]:
corpus_data_dir = CORPUS_DIR / "data"
corpus_data_dir.mkdir(parents=True, exist_ok=True)

metadata_path = CORPUS_DIR / "merlin_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump({"corpus_id": CORPUS_ID, "class": "ColPaliDataset"}, f)

corpus_columns = ["image", "image_filename", "docid", "source", "text"]
converted_corpus_ds = corpus_ds.select_columns(corpus_columns)
assert "source" in converted_corpus_ds.column_names

for shard_idx in tqdm(range(CORPUS_NUM_SHARDS), desc="Writing corpus parquet shards"):
    shard = converted_corpus_ds.shard(num_shards=CORPUS_NUM_SHARDS, index=shard_idx, contiguous=True)
    shard_path = corpus_data_dir / f"train-{shard_idx:05d}-of-{CORPUS_NUM_SHARDS:05d}.parquet"
    shard.to_parquet(str(shard_path))

print(f"Wrote corpus metadata: {metadata_path}")
print(f"Wrote {CORPUS_NUM_SHARDS} corpus parquet shards under: {corpus_data_dir}")

#### Saving the train set (queries, positives, negatives)

In [ ]:
converted_examples = []
missing_doc_ids = set()

for row in tqdm(query_ds, total=len(query_ds), desc="Converting queries"):
    positive_doc_ids = normalize_doc_ids(row["positive_document_ids"])
    negative_doc_ids = normalize_doc_ids(row["negative_document_ids"])

    pos_doc, missing_pos_doc_ids = map_document_ids(positive_doc_ids, docid_to_image_filename)
    neg_doc, missing_neg_doc_ids = map_document_ids(negative_doc_ids, docid_to_image_filename)
    missing_doc_ids.update(missing_pos_doc_ids)
    missing_doc_ids.update(missing_neg_doc_ids)

    converted_examples.append(
        {
            "question_id": f"q{row['query_id']}",
            "question": str(row["query_text"]),
            "corpus_id": CORPUS_ID,
            "pos_doc": pos_doc,
            "neg_doc": neg_doc,
        }
    )

if missing_doc_ids:
    sample_missing = sorted(missing_doc_ids)[:20]
    raise ValueError(
        f"{len(missing_doc_ids):,} query document id(s) were not found in the corpus. "
        f"Sample missing ids: {sample_missing}"
    )

empty_positive_examples = [example["question_id"] for example in converted_examples if not example["pos_doc"]]
if empty_positive_examples:
    raise ValueError(f"Examples with empty pos_doc: {empty_positive_examples[:20]}")

training_json = {
    "corpus": {"path": str(CORPUS_DIR.resolve())},
    "data": converted_examples,
}

with TRAIN_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(training_json, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(converted_examples):,} training examples to: {TRAIN_JSON_PATH}")

#### Checking the converted train set and corpus

In [ ]:
with TRAIN_JSON_PATH.open("r", encoding="utf-8") as f:
    written_training_json = json.load(f)

assert written_training_json["corpus"]["path"] == str(CORPUS_DIR.resolve())
assert len(written_training_json["data"]) == len(query_ds)
assert (CORPUS_DIR / "merlin_metadata.json").is_file()
assert len(list((CORPUS_DIR / "data").glob("train-*.parquet"))) == CORPUS_NUM_SHARDS

sample_example = written_training_json["data"][0]
print(f"Training examples: {len(written_training_json['data']):,}")
print(f"Corpus documents: {len(docid_to_image_filename):,}")
print(f"Training JSON: {TRAIN_JSON_PATH}")
print(f"Corpus folder: {CORPUS_DIR}")
print(json.dumps(sample_example, indent=2, ensure_ascii=False)[:2000])

In [ ]:
from nemo_automodel.components.datasets.llm import make_retrieval_dataset

smoke_dataset = make_retrieval_dataset(
    data_dir_list=str(TRAIN_JSON_PATH),
    data_type="train",
    n_passages=5,
    do_shuffle=False,
)

sample = smoke_dataset[0]
assert isinstance(sample["question"], str) and sample["question"]
assert len(sample["doc_id"]) == 5
assert len(sample["doc_image"]) == 5
assert sample["doc_image"][0] != ""

print("AutoModel retrieval dataset smoke test passed")
print(f"Question: {sample['question']}")
print(f"Document ids: {sample['doc_id']}")
print(f"Positive image size: {sample['doc_image'][0].size}")